In [10]:
from hyper_fingerprints import Encoder, cosine_similarity

enc = Encoder(dimension=1024, seed=42)

# Encode molecules (SMILES strings or RDKit Mol objects)
fps = enc.encode(["CCO", "CO", "c1ccccc1"])  # shape: (3, 512), dtype: float64

# Cosine similarity — similar molecules get similar vectors
sim = cosine_similarity(fps, fps)

print(f"ethanol vs methanol: {sim[0, 1]:.3f}")   # high similarity
print(f"ethanol vs benzene:  {sim[0, 2]:.3f}")    # low similarity

ethanol vs methanol: 0.500
ethanol vs benzene:  0.001


In [5]:
fps

array([[ 1.93654973e+01,  5.86860903e+00,  7.85436234e-02, ...,
         1.34458681e+01,  2.06711739e+01,  1.83928767e+01],
       [-1.18432856e+00, -6.13431853e-01, -1.51530587e-01, ...,
        -1.43099700e+00,  1.12019141e+00,  1.63636630e+00],
       [-4.74613993e+03,  2.86745328e+03, -1.24669748e+03, ...,
        -3.93352385e+02, -1.03054739e+03,  4.37632437e+02]],
      shape=(3, 512))

In [3]:
import numpy as np
enc = Encoder(dimension=1024, seed=42)

joint = enc.encode_joint("CCO")
print(f"Joint shape: {joint.shape}")  # (1, 512) = 2 * 256

order_0 = joint[:, :enc.dimension]
order_n = joint[:, enc.dimension:]

# The second half matches encode()
fp = enc.encode("CCO")
print(f"Order-N matches encode(): {np.allclose(order_n, fp)}")

Joint shape: (1, 2048)
Order-N matches encode(): True


Calculate HDF for Toxcast 

In [3]:
import pandas as pd
toxcast_smiles = pd.read_csv(r"Q:\Messdaten\EXPECTmine_data\02_MLinvitroTox_PosNeg\mlinvitrotox\data\processed\chemical_fingerprints\ToxCast_True_FPS_filtered.csv")
toxcast_smiles.head(2)

,DTXSID,smiles,21,33,34,36,39,40,48,58,...,17796,17835,17839,17850,17852,17858,17868,17927,17945,17987
0,DTXSID00190950,OCC(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)F,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,DTXSID00192353,O=S(=O)(O)CCC(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F...,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
toxcast_smiles.shape

(8840, 3523)

In [7]:
from hyper_fingerprints import Encoder
import numpy as np

X = []
valid_idx = []

for i, smi in enumerate(toxcast_smiles['smiles']):
    try:
        X.append(enc.encode(smi))
        valid_idx.append(i)
    except Exception as e:
        print(f"Failed on row {i}: {smi}")

X = np.vstack(X)
toxcast_valid = toxcast_smiles.iloc[valid_idx].reset_index(drop=True)

Failed on row 128: C[Si](Cl)(Cl)CCC(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)F
Failed on row 303: CO[SiH](OC)OC
Failed on row 308: CC(=O)O[Si](C)(OC(C)=O)OC(C)=O
Failed on row 328: CCCC[Sn](Cl)(Cl)Cl
Failed on row 459: CO[Si](C)(C)OC
Failed on row 495: [Hg+]c1ccccc1
Failed on row 503: CCC[Si](OC(C)=O)(OC(C)=O)OC(C)=O
Failed on row 559: O=C1OC2(c3cc(Br)c([O-])cc3Oc3c2cc(Br)c([O-])c3[Hg+])c2ccccc21
Failed on row 679: [Yb+3]
Failed on row 830: Cl[Dy](Cl)Cl
Failed on row 845: C[As+](C)(C)CC(=O)[O-]
Failed on row 965: c1ccc([Sn+](c2ccccc2)c2ccccc2)cc1
Failed on row 1148: *CC(*)(C)C(=O)OCC(C)O
Failed on row 1179: C[Si]1(C)O[Si](C)(C)O[Si](C)(C)O[Si](C)(C)O[Si](C)(C)O1
Failed on row 1190: C[Sn](Cl)(Cl)Cl
Failed on row 1214: *CCCCCCCCCCCC
Failed on row 1232: Cl[Sn](Cl)(Cl)Cl
Failed on row 1287: O[Si](c1ccccc1)(c1ccccc1)c1ccccc1
Failed on row 1308: C[SiH](O[Si](C)(C)C)O[Si](C)(C)C
Failed on row 1324: COCCOCCC[Si](C)(O[Si](C)(C)C)O[Si](C)(C)C
Failed on row 1371: CCCCC[Si](OCC)(OCC)OCC
Failed on r

In [12]:
# Create feature columns named "0", "1", ..., "1023"
fingerprint_df = pd.DataFrame(
    X,
    columns=[str(i) for i in range(X.shape[1])]
)

# Add chemical identifier column
fingerprint_df.insert(0, 'DTXSID', toxcast_valid['DTXSID'].values)


In [13]:
fingerprint_df

,DTXSID,0,1,2,3,4,5,6,7,8,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,DTXSID00190950,9523.900542,1134.142689,-2212.919711,6511.149873,-20332.908477,3508.011969,9581.632111,-6403.687238,9556.213730,...,-5192.237537,-5369.773886,10388.524988,-14862.748322,3254.765011,9787.066190,-4677.086401,6596.904264,-3813.410393,-10480.931607
1,DTXSID00192353,14649.254470,1301.619395,-3066.983721,9548.262508,-31041.072785,5683.204941,13929.355384,-10470.392724,15085.115346,...,-8680.706885,-7094.279723,15251.824557,-22810.014888,5231.607869,14288.944473,-7401.481067,10886.188512,-6120.196874,-15355.253305
2,DTXSID00194615,12105.112821,99.889403,-2344.827925,7950.833388,-24850.309187,4462.151598,11276.406561,-8942.250695,12607.715906,...,-8223.020201,-4778.701833,12613.069590,-18834.773946,4621.722705,10961.482275,-6681.775156,10127.712846,-4862.260830,-12235.872239
3,DTXSID0020020,-323.489225,169.805366,162.052873,-195.340545,349.206806,134.451696,190.680820,49.840925,192.999654,...,172.237420,-34.145468,198.678945,-446.874838,261.108932,-399.874203,-309.942480,201.142918,-420.260655,-67.260579
4,DTXSID0020022,-1232.528111,-424.504730,-416.237768,-622.935060,-440.493629,-195.347328,981.808193,514.168570,-38.954704,...,209.524341,321.607424,965.790040,385.542803,370.305138,468.174473,320.204467,128.404966,-655.340551,-414.084891
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8670,DTXSID90868151,23603.010118,2598.036068,-5317.835424,15136.931361,-50502.571476,8633.751533,21768.175336,-17042.397379,24680.935521,...,-14309.838226,-10149.585790,23070.352067,-36887.589260,9206.963077,22941.877960,-11684.874450,18137.026943,-9954.019110,-24361.342769
8671,DTXSID90880131,2553.783279,-886.277695,-257.628648,2186.557848,-4754.690266,1248.328757,2758.019682,-2138.488580,2806.240450,...,-2059.087848,-1356.243048,3409.168079,-4038.629697,982.721923,2273.036471,-1747.708393,2473.074551,-937.214054,-2691.169977
8672,DTXSID90880156,12316.060497,1024.997630,-2567.661023,8086.912640,-25915.366137,4829.318406,11756.829021,-8686.030143,12611.507469,...,-7057.910511,-6114.238575,12932.881125,-19023.183212,4299.179894,12086.779827,-6071.982259,8888.626977,-5103.713609,-12947.520583
8673,DTXSID90891560,-4290.588585,2373.519519,-2750.313164,2292.997432,301.023389,-162.393278,-1558.941360,1775.209481,695.077539,...,-528.745174,412.382554,-877.282556,1231.263404,-1924.194815,-177.912575,-3037.780459,2719.118222,-3697.107063,2178.322879


In [14]:
fingerprint_df.to_csv(r"Q:\Messdaten\EXPECTmine_data\02_MLinvitroTox_PosNeg\mlinvitrotox\data\processed\chemical_fingerprints\HDF_1024_fingerprints.csv",index=False)